# Iterative Prisoner's Dilemma


### Description

The [Prisoner's Dilemma](https://en.wikipedia.org/wiki/Prisoner%27s_dilemma) (PD) is a classical game analyzed in game theory, which is widely used to (attempt to) model social/economical interaction. It's a "dilemma" as, if exploited to explain the emergence of altruism in human or in general animal society, it fails badly at a first glance.

The classical situation-representation of the PD is that of two prisoners whose conviction depends on their mutual cooperation. It is easier understood though if illustrated in terms of a trade-off game (closed bag exachange):

*Two people meet and exchange closed bags, with the understanding that one of them contains money, and the other contains a purchase. Either player can choose to honor the deal by putting into his or her bag what he or she agreed, or he or she can defect by handing over an empty bag.*

It is obvious that for both players the winning strategy is to NOT cooperate.

Things changes when the interaction between the two individuals is iterated, in that case a more altruist attitude (strategy) is expected to emerge. The goal of this project is to test this hypothesis.

Mathematically the PD can be expressed with very basic linear algebra. The key component is the **Payoff matrix** $M$, which quantify the reward each player gets depending on whether she cooperated or not (defect):

$$
M =
\begin{pmatrix}
R & S \\
T & P
\end{pmatrix}
$$

with $T,R,S,P$ integers that satisfy the following conditions:

$$
T>R>P>S; \quad 2R > T+S
$$

for example $T=3$, $R=2$, $P=1$ and $S=0$, or  $T=5$, $R=3$, $P=2$, $S=0$. Each player choice (move) can be represented by one of the two axis in ${\rm I\!R}^2$, i.e. $u_C=\begin{pmatrix} 1 \\ 0 \end{pmatrix}$ or $u_D=\begin{pmatrix} 0 \\ 1 \end{pmatrix}$, where the first coordinate stands for *Cooperate* and the second for *Defect*. Being $u_1$ and $u_2$ their rewards $r_1$ and $r_2$ can be computed then as:

$$
r_1 = u_1^T M u_2
\quad
\quad
r_2 = u_2^T M u_1
$$

In an Iterative Prisoner's Dilemma (IPD), two players play prisoner's dilemma more than once in succession and they remember previous actions of their opponent and change their strategy accordingly. The winning strategy is the one which yields to a larger reward at the end of the IPD.

The strategy can be represented as a function which outputs either $u_C$ or $u_D$. Such function can depend on the opponent's history of moves, her on history of moves, on the number of moves played till that moment and so on, but it can only be based on a probability density function. Possible strategies are:

* **Nice guy**: always cooperate (the function's output is always $u_D$)
* **Bad guy**: always defect
* **Mainly nice**: randomly defect $k\%$ of the times and cooperate $100-k\%$, $k<50$
* **Mainly bad**: randomly defect $k\%$ of the times and cooperate $100-k\%$, $k>50$
* **tit-for-tat**: start by cooperating, then repeat what the opponent has done in the previous move

Many more and much more complex strategies can be implemented. The strategy can even change during the IPD.


### Assignments

* Implement a simple IPD between two players implementing two given strategies. Study the evolution along the tournament confronting different strategies; study the overall outcome in the different configurations.
* Implement a multiple players IPD (MPIPD) where several strategies play against each other in a roud-robin scheme
* Iterate what done in the previous task (repeated MPIPD, rMPIPD)  by increasing the population implementing a given strategy depending on the results that strategy achieved in the previous iteration
* (*difficult*) Implement a rMPIPD where strategies are allowed to mutate. The goal is to simulate the effect of genetic mutations and the effect of natura selection. A parameter (gene) should encode the attidue of an individual to cooperate, such gene can mutate randomly and the corresponding phenotype should compete in the MPIPD such that the best-fitted is determined.  




---



---



# **The Prisoner's Dilemma: The Emergence of Cooperation**
Is it possible for cooperation to emerge in any environment without central authority?

This question plays a major role in many disciplines, such as psychology, biology, economics and political science, and applies to important problems like the *Security Dilemma*: nations often seeks their own security through means which challenge the security of others.

Historically, philosophers gave a lot of answer to this question. Thomas Hobbes thought that the state of nature was dominated by selfish individuals, while Jean-Jacques Rousseau had basically a complete different point of view on the matter.

Despite the interest one can pay to philosopy, we should ask wheter it exists a rigorous mathematical that can help us to answer the question.

Maybe the simplest model avaible is The Two Prisoner's Dilemma.
In this game, there are two players. Each has to decide wheter to defect or cooperate on the other prisoner without knowing what the other will do.



Two people meet and exchange closed bags, they could either give to the other an empty bag or a bag which contains money. Each prisoner can have it's own personality, he can be nice and always cooperate, mid, defecting with some probability, bad and always defecting, tit-for-tat, who starts by cooperating than repeat what the opponent has done in the previous move.
In the code the reward is handled by a payoff matrix. It rewards individual
                                                                  
defection.
From the point of view of one prisoner:

-he cooperates and the even the other does it: he gets 3

-he cooperates and the other doesn't: he gets 0

-he defeat and the other cooperate: he gets 5

-they both defeat: he gets 2


In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import matplotlib.animation as animation
from matplotlib.ticker import AutoMinorLocator, MultipleLocator, FuncFormatter
import seaborn as sns
import scipy as scp
import random
from matplotlib.patches import Circle
from matplotlib.patches import Patch
from IPython.display import HTML
import matplotlib.cm as cm
import time
import pickle

from numpy.typing import NDArray
from __future__ import annotations
from IPython.display import HTML


# Class Definition

At the beginning is important to initialize a class prisoner that defines the variables and the methods that allow its description and how he behaves:

* The dictionary with all the types of personality, the dictionary with all the possible actions, the payoff matrix

* The init method that initializes the prisoner by assinging a personality type, a parameter k, initializing the memory and the reward variables

* The action method which describes if the prisoner will cooperate or defect depending on its personality and the k parameter

* The interaction method computes the rewards obtained by two prisoners when they interact

* The update memory method records the last action made by an opponent, allowing the tit-for-tat personality to choose what he's going to do the next time.

In [ ]:
# @title
class Prisoner():

    personality_dict = {'Nice': 0, 'Mid': 1, 'Bad': 2, 'Tit-tat': 3}
    inv_dict         = {0: 'Nice', 1: 'Mid', 2: 'Bad', 3: 'Tit-tat'}

    action_dict = {
        'Cooperate': np.array([1, 0]),
        'Defect':    np.array([0, 1])
    }

    payoff_matrix = np.array([[3, 0],
                               [5, 2]])

    general_id = 0

    def __init__(self, personality_par: str, defect_percentage: float):
        if personality_par == 'Mid':
              if defect_percentage <= 0:
                  personality_par = 'Nice'
              elif defect_percentage >= 1:
                  personality_par = 'Bad'

        self.personality    = Prisoner.personality_dict[personality_par]
        self.k_par          = {'Bad': 1, 'Nice': 0, 'Tit-tat': 0}.get(personality_par, defect_percentage)
        self.memory         = {}
        self.total_reward   = 0
        self.single_reward  = []
        self.id             = Prisoner.general_id
        Prisoner.general_id += 1

        self.hist_x, self.hist_y = [], []
        self.trace = None

    #to handle the case k_par and personality is mismatched
        if personality_par == 'Bad':
          self.k_par = 1
        elif personality_par == 'Nice':
          self.k_par = 0
        #the following is commented to let tit-tat have k-par in thir genes
        #for a more complex evolution
        """
        elif personality_par == 'Tit-tat':
          self.k_par = 0
        """
    def action(self, opponent_id) -> NDArray:                       #  NDArray's shape (2,)
        if self.personality == Prisoner.personality_dict['Nice']:
            return Prisoner.action_dict['Cooperate']

        elif self.personality == Prisoner.personality_dict['Bad']:
            return Prisoner.action_dict['Defect']

        elif self.personality == Prisoner.personality_dict['Mid']:
            if self.k_par > np.random.rand():
                return Prisoner.action_dict['Defect']
            else:
                return Prisoner.action_dict['Cooperate']

        else:  # Tit-for-tat
            if opponent_id not in self.memory:
                return Prisoner.action_dict['Cooperate']
            return self.memory[opponent_id]

    def update_memory(self, action: NDArray, opponent_id: int):
        self.memory[opponent_id] = action

    # Average reward
    @property
    def average_reward(self):
        return self.total_reward / len(self.single_reward) if self.single_reward else 0

    # Interazione unica - Vecchia versione 2
    @staticmethod
    def interaction(p1: 'Prisoner', p2: 'Prisoner'):
        u1 = p1.action(p2.id)
        u2 = p2.action(p1.id)

        r1 = u1.dot(Prisoner.payoff_matrix.dot(u2.T))
        r2 = u2.dot(Prisoner.payoff_matrix.dot(u1.T))

        p1.single_reward.append(r1)
        p2.single_reward.append(r2)

        p1.total_reward += r1
        p2.total_reward += r2

        p1.update_memory(u2, p2.id)
        p2.update_memory(u1, p1.id)

        return r1, r2

    # for a deep copy of a prisoner
    def clone(self):
        new = Prisoner(self.inv_dict[self.personality], self.k_par)
        new.total_reward = self.total_reward

        return new



# Utility functions

## **Creating environments**

In [ ]:
# @title
def nonRandom_env_generator(players_characteristics : list,
                         population_size : int) -> list:
    """"
    This function gets the population size and a list of characteristics.
    It gives back a list of Prisoners distributed according to their percentage
    -----------------------------------------------------------------
    Parameters
    ----------
    players_characteristics : list of lists as following
                              [ [percentage, personality, defect_percentage], ... ]
                              each inner list represent one type of prisoner
    population_size         : number of total prisoners

    Returns
    -------
    players                  : list of Prisoners distributed according to their percentage
    """

    counts = [int(population_size*chrt[0]) for chrt in players_characteristics]
    # list with number of each type of player

    players = []

    for count, params in zip(counts, players_characteristics):
        for _ in range(count):
          players.append(Prisoner(params[1], params[2]))

    return players


def random_env_generator(players_characteristics : list,
                         population_size : int) -> list:
    """"
    This function gets the population size and a list of characteristics.
    It gives back a list of Prisoners distributed according to their percentage
    using multinomial distribution.
    This should be used when you call characteristics_list_generator() function
    with some pdf from SCIPY library.
    -----------------------------------------------------------------
    Parameters
    ----------
    players_characteristics : list of lists as following
                              [ [percentage, personality, defect_percentage], ... ]
                              each inner list represent one type of prisoner
    population_size         : number of total prisoners

    Returns
    -------
    players                  : list of Prisoners distributed according to their percentage
    """

    probability_list = [ chrt[0] for chrt in players_characteristics ]

    counts = np.random.multinomial(population_size, probability_list)

    players = []

    for count, params in zip(counts, players_characteristics):
        for _ in range(count):
          players.append(Prisoner(params[1], params[2]))

    return players


def characteristics_list_generator(tit_tat_probability : float,
                                   pdf : callable,
                                   *args, **kwargs) -> list:
    """
    This function helps you to create the list used in 'random_env_generator()'
    You must provide the probability for tit-tat prisoners
    and a pdf from SCIPY library
    -----------------------------------------------------------------
    Parameters
    ----------
    tit_tat_probability : float
    pdf                 : callable
    *args               : arguments for pdf
    **kwargs            : keyword arguments for pdf

    Returns
    -------
    characteristics     : list of lists as following
                          [ [percentage, personality, defect], ...]
                          each inner list represent one type of prisoner
    """

    k_grid = 1000
    k = np.linspace(-1,2,k_grid)

    prob = pdf(k, *args, **kwargs)
    prob /= np.sum(prob)
    prob *= 1 - tit_tat_probability

    characteristics = []

    for i in range(k_grid):
        characteristics.append([prob[i], "Mid", k[i]])

    characteristics.append([tit_tat_probability, "Tit-tat", 0])

    return characteristics


## **Round Robin functions**

In [ ]:
def round_robin(players : list, round_number : int) -> NDArray:
  """
    Simulates a round-robin with the given prisoners for a fixed number of rounds
    Returns the whole 3D matrix payoff, in which each 2D matrix is one round
    Each 2D matrix looks like:

       -           match_1                    match_2           ...         match_n
    player_1  cumul-reward_1(match_1)  cumul-reward_1(match_2)  ...  cumul-reward_1(match_n
    player_2  cumul-reward_2(match_1)  cumul-reward_2(match_2)  ...  cumul-reward_2(match_n
    ...              ...                       ....             ...          ...
    player_n  cumul-reward_n(match_1)  cumul-reward_n(match_2)  ...  cumul-reward_n(match_n
    -----------------------------------------------------------------------------
    Parameters:
    ----------
    players : list of Prisoners
    round_number : int

    Returns:
    --------
    reward_vs_iteration : 3D array
  """

  reward_vs_iteration = np.zeros((round_number, len(players), len(players)))
  actions = np.zeros((round_number, len(players), len(players)))

  players_copy = [p.clone() for p in players]
  n = len(players)
  for round in range(round_number):
    for match_num in range(n):
      for i in range(n):
        j = (i + match_num) % n
        r1, r2 = Prisoner.interaction(players[i], players_copy[j])
        reward_vs_iteration[round, i, match_num] = players[i].total_reward
        actions[round, i, match_num] = r1



  return np.array(reward_vs_iteration), np.array(actions)


## Animazioni

In [ ]:
# @title
def rotating_players(
    fig, ax, data: np.ndarray,
    color : list,
    total_duration_ms: int = 5000,
    moving_time=5, stop_time=15):
    """
    takes a 3D array of shape (n_rounds, n_players, n_players) generated by
    the utility function payoff_to_action_matrix() and generates the animation
    of the players' actions

    Parameters
    ----------
    fig, ax           : matplotlib Figure e Axes
    data              : np.ndarray shape (n_rounds, n_players, n_players)
                        as returned by payoff_to_action_matrix()()
    colors            : colors for the players
    total_duration_ms : duration of the animation in milliseconds
    moving_time       : number of frames for the balls to move
    stop_time         : number of frames for the balls to stop

    Returns
    -------
    update            : function to update the animation
    all_artists       : list of all artists to be updated
    """

    n_rounds, n_players, _ = data.shape

    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_aspect('equal')
    ax.axis('off')

    #-------------------------------------------
    # Defining the parameters for balls position
    #--------------------------------------------
    #starting from pi/2 so that the first player is in top position
    angles = np.linspace(0.5*np.pi, 2.5*np.pi, n_players, endpoint=False)
    delta_a = angles[1] - angles[0]

    r_inner = 2
    r_outer = 4

    colors = ['gray'] * n_players

    inner_circles = []
    outer_circles = []
    arrows = []

    #-----------------------------------------
    #         Inizialiing balls
    #-----------------------------------------
    # definining internal circles and append them to the inner_circles list
    for i in range(n_players):
        x = r_inner * np.cos(angles[i])
        y = r_inner * np.sin(angles[i])
        c = Circle((x, y), 0.5, color=colors[i])
        ax.add_patch(c)
        inner_circles.append(c)

    # definining outer circles and append them to the outer_circles list
    for i in range(n_players):
        x = r_outer * np.cos(angles[i])
        y = r_outer * np.sin(angles[i])
        c = Circle((x, y), 0.5, color=colors[i])
        ax.add_patch(c)
        outer_circles.append(c)

    # if defined, using personalized edgecolors
    if len(color) > 0:
        if len(color) < n_players:
            print("INVALID COLOR LIST: not using it!")
        for i, (circle_out, circle_in) in enumerate(zip(outer_circles,
                                                        inner_circles)):
            circle_out.set_edgecolor(color[i])
            circle_in.set_edgecolor(color[i])
            circle_out.set_lw(2)
            circle_in.set_lw(2)
            circle_out.set_zorder(3)
            circle_in.set_zorder(3)

    #-----------------------------------
    #    Initializing interaction arrows
    #-----------------------------------
    for i in range(n_players):
        angle = angles[i]

        #ending point of the arrow
        x = (r_outer - 0.4) * np.cos(angle)
        y = (r_outer - 0.4) * np.sin(angle)
        end = (x, y)

        #starting point of the arrow
        x = (r_inner + 0.4) * np.cos(angle)
        y = (r_inner + 0.4) * np.sin(angle)
        start = (x, y)

        arrow = ax.annotate("", xy=end, xytext=start,
                            arrowprops=dict(arrowstyle="<->", lw=3,
                                            color="gray", alpha=0.7))
        arrow.set_visible(True)
        arrows.append(arrow)

    #----------------------------------------
    #  Creating text cell for round counter
    #----------------------------------------
    text = ax.text(0, 0, "Round 1", animated=True, alpha=0.7)
    text.set_horizontalalignment('center')
    text.set_verticalalignment('center')
    text.set_fontweight('roman')
    text.set_size(23)
    text.set_zorder(0)

    #-------------------------------------
    #    Initializing the legend
    #------------------------------------
    coop_color = (0, 1, 0, 0.6) #green with alpha 0.6
    defect_color = (1, 0, 0, 0.6) #red with alpha 0.6

    legend_elements = [
        Patch(facecolor=coop_color, label="Cooperate",
              edgecolor=(0,0,0,0.7),  linewidth=2),
        Patch(facecolor=defect_color, label="Defect",
              edgecolor=(0,0,0,0.7), linewidth=2)
    ]

    legend = ax.legend(handles=legend_elements, loc="upper center",
                       bbox_to_anchor=(0.5, 0.0),frameon=False, ncols=2,
                       fontsize='large', labelcolor=(0,0,0,0.7))
    legend.set(zorder=0)

    #----------------------------------
    #      Update function
    #---------------------------------
    #variable assigment before the first update
    angles_outer = angles.copy()

    is_rotating = False
    counter_rot = 0
    counter_stop = 0
    interaction_idx = 0
    round_idx = 0

    all_artists = outer_circles + inner_circles + arrows + [text]



    def update(frame):
        nonlocal is_rotating, counter_rot, counter_stop
        nonlocal angles_outer, interaction_idx, round_idx

        # what to do if the circles are moving around
        if is_rotating:
            counter_rot += 1
            rotation = counter_rot * delta_a / moving_time

            #moving outer cirlces
            for i, circle in enumerate(outer_circles):
                angle = angles_outer[i] + rotation
                x = r_outer * np.cos(angle)
                y = r_outer * np.sin(angle)
                circle.center = (x, y)

            #check end of rotation condition and setting stop mode
            if counter_rot == moving_time:
                angles_outer += delta_a
                is_rotating = False
                counter_rot = 0
                for arrow in arrows:
                    arrow.set_visible(True)

        else:
            # what to do inf the circles are stopping
            n = len(inner_circles)
            if counter_stop == 0 and round_idx < n_rounds:
                for i in range(n):
                    j = interaction_idx

                    actions = data[round_idx, i, j]
                    #both cooperate
                    if actions == Prisoner.payoff_matrix[0][0]:
                        outer_circles[(i+j)%n].set_facecolor(coop_color) #green with alpha 0.6
                        inner_circles[i].set_facecolor(coop_color)
                    #the prisoner cooperates, the opponent defects
                    elif actions == Prisoner.payoff_matrix[0][1]:
                        outer_circles[(i+j)%n].set_facecolor(defect_color) #red with aplha 0.6
                        inner_circles[i].set_facecolor(coop_color)
                    #the prisoner defects, the opponent cooperates
                    elif actions == Prisoner.payoff_matrix[1][0]:
                        outer_circles[(i+j)%n].set_facecolor(coop_color)
                        inner_circles[i].set_facecolor(defect_color)
                    #both defect
                    else:
                        outer_circles[(i+j)%n].set_facecolor(defect_color) #red with aplha 0.6
                        inner_circles[i].set_facecolor(defect_color) #red with aplha 0.6

                interaction_idx += 1

                #check if the round robin is finished
                if interaction_idx == n_players:
                    interaction_idx = 0
                    round_idx += 1


            counter_stop += 1

            #check end of stop time condition and setting rotation mode
            if counter_stop == stop_time:
                is_rotating = True
                counter_stop = 0
                if interaction_idx == 0:
                    text.set_text(f"Round {round_idx+1}")

                # reset the body color to gray when the balls are spinning
                for circle in outer_circles:
                    circle.set_facecolor("gray")
                    circle.set_alpha(0.6)
                for circle in inner_circles:
                    circle.set_facecolor("gray")
                    circle.set_alpha(0.6)
                for arrow in arrows:
                    arrow.set_visible(False)

        return all_artists


    return update, all_artists

In [ ]:
# @title
def reward_time_animation(
    fig, ax, data: np.ndarray,
    colors: list, names: list,
    moving_time: int = 5, stop_time: int = 15,
    trail_lw: float = 2.5, trail_alpha: float = 0.6, dot_size: float = 8,
):
    """
    takes a 3D array of shape (n_rounds, n_players, n_players) generated by
    the utility function round_robin() and generates the animation match number
    versus cumulative reward

    Parameters
    ----------
    fig, ax           : matplotlib Figure e Axes
    data              : np.ndarray shape (n_rounds, n_players, n_players)
                        as returned by round_robin()
    colors            : colors for the players
    names             : names for the players
    moving_time, stop_time    : to be set as the same in rotating players to have
                              : the animation sincronized
    trail_lw          : trail width
    trail_alpha       : alpha for the trail
    dot_size          : size of the dot representing the player

    Returns
    -------
    update            : function to update the animation
    all_artists       : list of all artists to be updated
    """
    n_rounds, n_players, _ = data.shape
    ticks_per_round = n_players
    total_ticks     = n_rounds * ticks_per_round
    frames_per_tick = moving_time + stop_time   # number of frames for each match
                                                #needed to be sincronized with
                                                #rotating_players()

    #----------------------------------------
    #           Preparing the data
    #----------------------------------------
    #gets from data the j-th column of the r-th matrix
    plot_data = np.zeros((total_ticks, n_players))
    for r in range(n_rounds):
        for j in range(n_players):
            plot_data[r * ticks_per_round + j] = data[r, :, j]

    #setting x labels (starting from one)
    x_global = np.arange(1, total_ticks + 1)

    #-------------------------------------
    #          Colored areas
    #-------------------------------------
    region_colors = ['#f2f2f2', '#ffffff']
    #for each round defines the starting and ending point of the round area
    #the edges of the area are located in the middle of two matches (thats why
    #0.5 is added)
    for r in range(n_rounds):
        # the parentesis are needed for handling
        x_start = r * ticks_per_round + (0.5 if r != 0 else 0)
        x_end   = x_start + ticks_per_round + (0.5 if r == 0 else 0)
        ax.axvspan(x_start, x_end, facecolor=region_colors[r%2], alpha=0.5, zorder=0)
        ax.axvline(x=x_start, color='gray', lw=0.8, ls='--', alpha=0.4, zorder=1)
        ax.text(x_start + ticks_per_round/2, 0, f'Round{r+1}',
                ha='center', va='bottom', fontsize=8, color='gray', alpha=0.7, zorder=2)
    #----------------------------------
    #   Initial setup for axis
    #----------------------------------
    ax.set_xlabel("Match number")
    ax.set_ylabel("Cumulative Reward")
    ax.set_xlim(0, ticks_per_round * 1.05)
    ax.set_ylim(0, 1)

    #---------------------------------
    # trails and dots setup
    # (representation of past and actual cumulated reward)
    #-------------------------------
    trails, dots = [], []
    for color, name in zip(colors, names):
        trail, = ax.plot([], [], lw=trail_lw, alpha=trail_alpha, color=color, label=name)
        dot,   = ax.plot([], [], 'o', ms=dot_size, color=color, zorder=5)
        trails.append(trail); dots.append(dot)

    #legend
    ax.legend(bbox_to_anchor=(-0.05, 1), loc='upper right', borderaxespad=0)

    all_artists = trails + dots

    def update(frame):

        if frame % frames_per_tick != 0:
            return all_artists

        tick = frame // frames_per_tick
        if tick >= total_ticks:
            return all_artists

        x_now = x_global[:tick + 1]
        for p_idx, (trail, dot) in enumerate(zip(trails, dots)):
            trail.set_data(x_now, plot_data[:tick+1, p_idx])
            dot.set_data([x_now[-1]], [plot_data[tick, p_idx]])

        current_round_end = (tick // ticks_per_round + 1) * ticks_per_round
        ax.set_xlim(0, current_round_end * 1.05)
        y_max = plot_data[:tick+1].max()
        ax.set_ylim(0, y_max * 1.15 if y_max > 0 else 1)

        return all_artists


    return update, all_artists

In [ ]:
# @title
def combine_animations(
    fig,
    update_fns: list,
    artists: list,
    total_frames: int,
    interval: int = 100,
):
    """
    Combines some functions to create a single animation

    Parameters
    ----------
    fig          : matplotlib Figure condivisa
    update_fns   : lista di funzioni update restituite dalle animazioni
    artists      : lista di liste di artisti restituiti dalle animazioni
    total_frames : numero totale di frame
    interval     : ms tra un frame e l'altro

    Returns
    -------
    animation.FuncAnimation
    """
    all_artists = [a for group in artists for a in group]

    def combined_update(frame):
        result = []
        for fn in update_fns:
            result += list(fn(frame))
        return result

    return animation.FuncAnimation(
        fig= fig,
        func= combined_update,
        frames= total_frames,
        interval= interval,
        blit= False,
    )

In [ ]:
# @title
def get_names_and_colors(prisoners: list)->tuple:
  """
  gets a prisoners list and returns the names and colors for the animation
  Parameters
  ----------
  prisoners : list of prisoners
  Returns
  -------
  names : list of names
  colors : list of colors
  """
  names = []
  colors = []
  for p in prisoners:
    names.append(p.inv_dict[p.personality])
    if names[-1] == "Nice":
      colors.append(cm.Greens(0.5))
    elif names[-1] == "Bad":
      colors.append(cm.Reds(0.8))
    elif names[-1] == "Mid":
      names[-1] += f"(defect : {p.k_par*100:.0f}%)"
      colors.append(cm.Blues(p.k_par))
    elif names[-1] == "Tit-tat":
      colors.append(cm.Oranges(0.5))
  return names, colors

## Evolution

In [ ]:
# @title
def evolution(players_list: list, round_number: int,
              best_fraction: float,
              keep_best=0.4,
              mutation=0.3,
              k_par_mutation_factor=0.1, personality_mutation_factor=0.1,
              heredity=0.3) -> list:
    """
    Generate a new generation of prisoners, given the actual generation, and a list
    of evolutionistic parameters to be setted
    ---------------------------------------------------------------------
    Parameters
    ----------
    players_list      : list of prisoners
    round_number      : number of rounds to be played in the generation
    best_fraction     : fraction of the best players to be used for creating the new generation
    keep_best         : fraction of the best_fraction to be kept unchanged
    mutation          : fraction of the best_fraction that randomly mutates
    k_par_mutation_factor : mutation intensity for defect parameter
    personality_mutation_factor : mutation intensity for personality change
    heredity          : fraction of the best_fraction that evolves with heredity
    ---------------------------------------------------------------------
    Returns
    -------
    new_generation    : list of prisoners
    """

    total_number     = len(players_list)
    survivors_number = max(int(total_number * best_fraction), 2)

    keep_best_n      = int(survivors_number * keep_best)
    mutant_number    = int(total_number * mutation)
    heredity_number  = int(total_number * heredity)
    #the following is used to keep the population size fixed
    filler_number    = total_number - keep_best_n - mutant_number - heredity_number

    if filler_number < 0:
        filler_number = 0
    rewards, _ = round_robin(players_list, round_number)
    # rewards shape: (round_number, n_players, n_players)
    # sum over opponents (axis=2) to get each player's total reward per round
    # then take the last round (axis=0 → [-1]) as the ranking metric
    final_rewards = rewards[-1].sum(axis=1)          # shape: (n_players,)

    best_ones = np.argsort(final_rewards)[::-1]
    survivors = [
                  Prisoner(Prisoner.inv_dict[players_list[i].personality],
                           players_list[i].k_par)
                for i in best_ones[:survivors_number]
                ]

    #keep_best
    new_generation = [survivors[i] for i in range(keep_best_n)]

    # mutate
    for i in range(mutant_number):
        p = random.choice(survivors)
        if random.random() < personality_mutation_factor:
            new_personality = Prisoner.inv_dict[random.choice([1, 3])]
        else:
            new_personality = Prisoner.inv_dict[p.personality]
        k = np.clip(p.k_par * (1 + np.random.normal(loc=0, scale=k_par_mutation_factor)), 0, 1)
        new_generation.append(Prisoner(new_personality, k))

    # heredity
    for i in range(heredity_number):
        p1, p2 = random.sample(survivors, 2)
        new_personality = Prisoner.inv_dict[random.choice([p1.personality, p2.personality])]
        k = np.clip((p1.k_par + p2.k_par) / 2, 0, 1)
        new_generation.append(Prisoner(new_personality, k))

    fillers = [ Prisoner(Prisoner.inv_dict[p.personality], p.k_par)
                for p in np.random.choice(survivors, filler_number)]
    new_generation += fillers

    return new_generation



In [ ]:
# @title
def simulate_evolution(prisoner_list_init, rounds_per_gen, generations_number,
                       best_fraction_func=None, keep_best=1,
                       mutation=0,
                       k_par_mutation_factor=0.1, personality_mutation_factor=0.1,
                       heredity=0.3):
    """
    Simulates the evolution of a population of prisoners over a number of generations.

    Parameters
    ----------
    prisoner_list_init   : initial list of prisoners
    rounds_per_gen       : number of round robins played each generation
    generations_number   : number of generations to simulate
    best_fraction_func   : function of t returning best_fraction (default: constant 0.5)
    keep_best            : fraction of survivors kept unchanged
    mutation             : fraction of population that mutates
    mutation_factor      : mutation intensity
    heredity             : fraction of population that is inherited

    Returns
    -------
    df : DataFrame with columns = strategy names, rows = generations (0 = initial)
    """
    if best_fraction_func is None:
        best_fraction_func = lambda t: 0.5

    prisoner_list = [p.clone() for p in prisoner_list_init]

    #gives back a the dict of the population number of each personality
    def get_gen(prisoner_list):
        gen = {}
        for p in prisoner_list:
            name = get_names_and_colors([p])[0][0] #[0][0] to get just names
                                                   #and taking the string
                                                   #(gives back a list of strings)
            gen[name] = gen.get(name, 0) + 1
        return gen

    personality_vs_gen = [get_gen(prisoner_list)]

    for t in range(generations_number):
        Prisoner.general_id = 0
        best_fraction = best_fraction_func(t)
        prisoner_list = evolution(prisoner_list, rounds_per_gen, best_fraction,
                                  keep_best=keep_best, mutation=mutation, heredity=heredity,
                                  k_par_mutation_factor=k_par_mutation_factor,
                                  personality_mutation_factor=personality_mutation_factor)
        personality_vs_gen.append(get_gen(prisoner_list))

    #if a personality disappears it should be equal to zero
    df = pd.DataFrame(data=personality_vs_gen).fillna(0)

    df.index.name = "generation"

    return df

In [ ]:
# @title
def k_par_mean_std_evolution(prisoner_list_init, rounds_per_gen, generations_number,
                       best_fraction_func=None, keep_best=1,
                       mutation=0,
                       k_par_mutation_factor=0.1, personality_mutation_factor=0.1,
                       heredity=0.3):
    """
    Simulates the evolution of a population of prisoners over a number of generations.
    -----------------------------------------------------------------------
    Parameters
    ----------
    prisoner_list_init   : initial list of prisoners
    rounds_per_gen       : number of round robins played each generation
    generations_number   : number of generations to simulate
    best_fraction_func   : function of t returning best_fraction (default: constant 0.5)
    keep_best            : fraction of survivors kept unchanged
    mutation             : fraction of population that mutates
    mutation_factor      : mutation intensity
    heredity             : fraction of population that is inherited

    Returns
    -------
    mean, stdev : mean and standard deviation of k_par for each generation
    """
    if best_fraction_func is None:
        best_fraction_func = lambda t: 0.5

    prisoner_list = [p.clone() for p in prisoner_list_init]

    mean = []
    stdev = []

    k_list = [p.k_par for p in prisoner_list]
    mean.append(np.mean(k_list))
    stdev.append(np.std(k_list))

    for t in range(generations_number):
        Prisoner.general_id = 0
        best_fraction = best_fraction_func(t)
        prisoner_list = evolution(prisoner_list, rounds_per_gen, best_fraction,
                                  keep_best=keep_best, mutation=mutation, heredity=heredity,
                                  k_par_mutation_factor=k_par_mutation_factor,
                                  personality_mutation_factor=personality_mutation_factor)
        k_list = [p.k_par for p in prisoner_list]
        mean.append(np.mean(k_list))
        stdev.append(np.std(k_list))

    return mean, stdev

# **1 - Iterated Prisoner's Dilemma**

Description of the simple IPD code:
The code computes ten interactions between two given personalities, it evaluates all the possible non-trivial combinations of personalities. We can observe that the bad personality always wins.
In partcular the reward between the bad and the tit-tat remains the same accross repeated encounters, meaning that the tit-for-tat can never beat bad in this set-up.

In [ ]:
rounds = 10

keys = list(Prisoner.personality_dict.keys())

fig, ax = plt.subplots(2, 4, figsize=(22, 6))
plt.close(fig)

match_data = []
plot_idx = 0
L = []

for i in range(4):
    for j in range(4):

        if (i != j and (i, j) not in L and (j, i) not in L) or (i == 1 and j == 1 and (i, j) not in L):

            p1 = Prisoner(keys[i], 0.65)
            p2 = Prisoner(keys[j], 0.65)

            L.append((i, j))

            for k in range(rounds):
                Prisoner.interaction(p1, p2)

            if p1.total_reward > p2.total_reward:
                winner = Prisoner.inv_dict[p1.personality]
            elif p2.total_reward > p1.total_reward:
                winner = Prisoner.inv_dict[p2.personality]
            else:
                winner = 'Draw'

            print(f'{keys[i]} vs {keys[j]} -> {winner}')

            y1 = np.cumsum(p1.single_reward)
            y2 = np.cumsum(p2.single_reward)
            x = np.arange(1, rounds + 1)

            if plot_idx < 4:
                c = 0
            else:
                c = 1

            axis = ax[c, plot_idx % 4]
            axis.set_facecolor('#f0f0f0')
            axis.set_xlim(1, rounds)
            axis.set_ylim(0, max(y1.max(), y2.max()) + 1)
            axis.set_title(f'{keys[i]} vs {keys[j]}')
            names, colors = get_names_and_colors([p1, p2])

            line1 = axis.plot([], [], '-', color=colors[0], lw=2)[0]
            line2 = axis.plot([], [], '-', color=colors[1], lw=2)[0]

            marker1 = axis.plot([], [], 'o', color=colors[0], markersize=8)[0]
            marker2 = axis.plot([], [], 'o', color=colors[1], markersize=8)[0]

            axis.legend([
                f'Player1: {Prisoner.inv_dict[p1.personality]}',
                f'Player2: {Prisoner.inv_dict[p2.personality]}'
            ])

            match_data.append((line1, line2, marker1, marker2, x, y1, y2))

            plot_idx += 1

ax[1, 3].axis("off")


def update(frame):
    artists = []

    for line1, line2, marker1, marker2, x, y1, y2 in match_data:

        line1.set_data(x[:frame], y1[:frame])
        line2.set_data(x[:frame], y2[:frame])

        if frame > 0:
            marker1.set_data([x[frame - 1]], [y1[frame - 1]])
            marker2.set_data([x[frame - 1]], [y2[frame - 1]])
        else:
            marker1.set_data([], [])
            marker2.set_data([], [])

        artists.extend([line1, line2, marker1, marker2])

    return artists


ani = animation.FuncAnimation(
    fig,
    update,
    frames=rounds + 1,
    interval=500,
    blit=True
)

for a in ax.flat:
    a.set_ylabel("Cumulative reward")

plt.tight_layout()

HTML(ani.to_jshtml())

$$
M =
\begin{pmatrix}
3 & 0 \\
5 & 2
\end{pmatrix}
$$

## Equations for the reward for each personality
For some personalities it is possible to study the reward given the number of interactions with another prisoner in a deterministic way:

**Bad vs Tit-tat**
$$ \text{r}_{Bad}(t) = 5 + 2*(t-1) $$
$$ \text{r}_{Tit-tat}(t) = 2*(t-1) $$

**Nice vs Tit-tat**
$$ \text{r}_{Nice}(t) = 3*t $$
$$ \text{r}_{Tit-tat}(t) = 3*t $$

**Bad vs Nice**
$$ \text{r}_{Bad}(t) = 5*t $$
$$ \text{r}_{Nice}(t) = 0  $$

If we study the behaviour against Mid personalities the topic becomes more complex: due to the random behaviour of the *Mid* personality the reward can be described as a stochastic process:

**Bad vs Mid**
$$ \text{r}_{Bad}(t) = 5*\mathcal{B}(t, 1-k) + 2*\mathcal{B}(t, k)   $$
$$ \text{r}_{Mid_k}(t) =  2*\mathcal{B}(t, k) + 0*\mathcal{B}(t,1−k) $$

**Good vs Mid**
$$ \text{r}_{Good}(t) = 3*\mathcal{B}(t, 1-k) + 0*\mathcal{B}(t,k)  $$
$$ \text{r}_{Mid_k}(t) =  5*\mathcal{B}(t, k) + 3*\mathcal{B}(t, 1-k)$$

**Tit-tat vs Mid**
$$ \text{r}_{Tit-Tat}(t) =  3*\mathcal{B}(1, 1-k) + 0*\mathcal{B}(1, k) + 5*\mathcal{B}(t-1, k*(1-k)) + 3*\mathcal{B}(t-1, (1-k)*(1-k)) + 2*\mathcal{B}(t-1, k*k) + 0*\mathcal{B}(t-1, (1-k)*k)$$
$$ \text{r}_{Mid_k}(t) =  3*\mathcal{B}(1, 1-k) + 5*\mathcal{B}(1, k) + 0*\mathcal{B}(t-1, k*(1-k)) + 3*\mathcal{B}(t-1, (1-k)*(1-k)) + 2*\mathcal{B}(t-1, k*k) + 5*\mathcal{B}(t-1, (1-k)*k)$$

Where $\mathcal{B}$ is the Bernoulli distribution, $k$ is the defect fraction for *Mid* prisoners and $t$ is the number of interactions.

As you can see, these formulas can be a mess. In order to visualize them better we want to study them in a computational way.

To visualize the reward obtained by different personalities when playing against another, we vary the parameter
k
k from 0 to 1. Each match is repeated 1,000 times to compute the mean and variance of the resulting rewards. We observe that the reward generally decreases as
k
k increases. At the beginning and end of the process, when
k=0
k=0 or
k=1
k=1, the variance is zero because the outcome is deterministic.

In [ ]:
k_opponent = np.linspace(0, 1, 20)
n_exp = 1000

p_label = ["Tit-tat", "Nice", "Bad"]
k_val = [0, 0, 1]

fig, AX = plt.subplots(1, 3, figsize=(20,5))

for l in range(len(p_label)):
  mean = np.zeros(len(k_opponent))
  std = np.zeros(len(k_opponent))
  for i in range(len(k_opponent)):
    for round in range(n_exp):
      prisoners = [ Prisoner(p_label[l], k_val[l]),
                    Prisoner("Mid", k_opponent[i])]

      data_rewards, data_actions = round_robin(prisoners, round_number=100)
      last_reward = data_rewards[-1,:,-1]
      mean[i] += last_reward[0]
      std[i] += last_reward[0]**2

    mean[i] /= n_exp
    std[i] = np.sqrt((std[i]/n_exp - mean[i]**2))



  color = get_names_and_colors([Prisoner(p_label[l], k_val[l])])[1][0]
  AX[l].fill_between(k_opponent, mean - std, mean + std, alpha=0.3, color=color)
  AX[l].plot(k_opponent, mean, label="Average Reward", color=color)
  AX[l].plot(k_opponent, mean - std, "k--", label="Population's stdev", alpha=0.3)
  AX[l].plot(k_opponent, mean + std, "k--", alpha=0.3)
  AX[l].set_xlabel("Opponent's defect fraction")
  AX[l].set_ylabel("Average Reward")
  AX[l].grid()
  AX[l].legend(loc='best')
  AX[l].set_title(f"{p_label[l]} vs Mid")

fig.tight_layout()
plt.show()

# **2 - Simulating Round Robin**

## What is a ***Round Robin***?
A round robin is a format in which every participant plays against every other participant, himself included. The following animation is a graphical representation of a six-player round robin, where each color represent a different player.

In [ ]:
# @title
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

n = 6

data_actions = np.random.randint(0, 2, (1, n, n))

cmap = plt.cm.get_cmap('tab10', n)
colors = [cmap(i) for i in range(n)]

update_balls,  artists_balls  = rotating_players(
    fig, ax, data_actions, color=colors,
    moving_time=4, stop_time=12
)

fig.tight_layout()

total_frames =16 * 6

ani = combine_animations(
    fig         = fig,
    update_fns  = [update_balls],
    artists     = [artists_balls],
    total_frames= total_frames,
    interval    = 100,
)

HTML(ani.to_jshtml())

## Multiple Players Iterative Prisoner's Dilemma

In this section a **Multiple Players Iterative Prisoner's Dilemma** (MPIPD) is implemented. We simply let several strategies play against each other in a round-robin scheme.

### Single Round case

###One player for each personality, different k-values for MID

In [ ]:
# @title
#definining prisoners
prisoners = [ Prisoner("Nice",0),
              Prisoner("Mid",0.25),
              Prisoner("Mid",0.75),
              Prisoner("Mid",0.5),
              Prisoner("Tit-tat",0),
              Prisoner("Bad",1) ]

data_rewards, data_actions = round_robin(prisoners, round_number=1)

#names and color for the animation
names, colors = get_names_and_colors(prisoners)

#animation
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

update_balls,  artists_balls  = rotating_players(
    fig, ax_left, data_actions, color=colors,
    moving_time=3, stop_time=9
)
update_reward, artists_reward = reward_time_animation(
    fig, ax_right, data_rewards, colors=colors, names=names,
    moving_time=3, stop_time=9
)

fig.tight_layout()

total_frames =12 * len(prisoners)

ani = combine_animations(
    fig         = fig,
    update_fns  = [update_balls, update_reward],
    artists     = [artists_balls, artists_reward],
    total_frames= total_frames,
    interval    = 150,
)

HTML(ani.to_jshtml())



Here the ranking for the the one round game:

In [ ]:
# @title
#Round Robin ranking
last_rewards = data_rewards[0,:,-1]
best_ones = np.argsort(last_rewards)[::-1]

names, _ = get_names_and_colors(prisoners)
print("The three best players are : ")
print('-'*30)
for i in best_ones[:3]:
  print(f"{i+1} :\t {names[i]},\t\t average final reward: {last_rewards[i]:.0f}")


### Which strategy wins on average in the One-Round MPIPD?
To understand this, we simulate a large number of single round robins to see which strategy gains the most reward in the end.
Here we have simulated 1000 single rounds.

NOTE: We reset the players before each round robin starts so that their memory is cleared; otherwise we would find ourselves in the multi-round MPIPD.

In [ ]:
# @title
n_exp = 1000
prisoners_number = 6

average_last_rewards = np.zeros(prisoners_number)
stdev_last_rewards = np.zeros(prisoners_number)

for round in range(n_exp):
  prisoners = [ Prisoner("Nice",0),
                Prisoner("Mid",0.25),
                Prisoner("Mid",0.75),
                Prisoner("Mid",0.5),
                Prisoner("Tit-tat",0),
                Prisoner("Bad",1) ]

  data_rewards, data_actions = round_robin(prisoners, round_number=1)
  last_rewards = data_rewards[0,:,-1]
  average_last_rewards += last_rewards
  stdev_last_rewards += last_rewards**2

average_last_rewards /= n_exp
stdev_last_rewards = np.sqrt( (stdev_last_rewards/n_exp
                              - average_last_rewards**2) )
best_ones = np.argsort(average_last_rewards)[::-1]

names, _ = get_names_and_colors(prisoners)
print("The three best players are : ")
print('-'*30)
for i in best_ones[:3]:
  print(f"{i+1} :\t {names[i]},\t\t average final reward: {average_last_rewards[i]}" +
        f"\tstdev: {stdev_last_rewards[i]}")



As in the Two-Player Prisoner's Dilemma, defecting is the dominant strategy.

Now we try again using other combination of strategies:
### NO TIT-TAT, normally distributed MID (mean = 0.5, stdev = 0.5)

In [ ]:
# @title
chrt = characteristics_list_generator(tit_tat_probability=0,
                                      pdf=scp.stats.norm.pdf,
                                      loc=0.5, scale=0.5)

n_prisoners = 10
prisoners = random_env_generator(chrt, n_prisoners)

data_rewards, data_actions = round_robin(prisoners, round_number=1)

#names and color for the animation
names, colors = get_names_and_colors(prisoners)

#animation
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

update_balls,  artists_balls  = rotating_players(
    fig, ax_left, data_actions, color=colors,
    moving_time=3, stop_time=9
)
update_reward, artists_reward = reward_time_animation(
    fig, ax_right, data_rewards, colors=colors, names=names,
    moving_time=3, stop_time=9
)

fig.tight_layout()

total_frames =12 * len(prisoners)

ani = combine_animations(
    fig         = fig,
    update_fns  = [update_balls, update_reward],
    artists     = [artists_balls, artists_reward],
    total_frames= total_frames,
    interval    = 150,
)

HTML(ani.to_jshtml())


In [ ]:
# @title
n_exp = 1000
prisoners_number = 10

average_last_rewards = np.zeros(prisoners_number)
stdev_last_rewards = np.zeros(prisoners_number)

for p in prisoners:
  p.total_reward=0

for round in range(n_exp):
  prisoners_exp = [ p.clone() for p in prisoners ]

  data_rewards, data_actions = round_robin(prisoners_exp, round_number=1)
  last_rewards = data_rewards[0,:,-1]
  average_last_rewards += last_rewards
  stdev_last_rewards += last_rewards**2

average_last_rewards /= n_exp
stdev_last_rewards = np.sqrt( (stdev_last_rewards/n_exp
                              - average_last_rewards**2) )
best_ones = np.argsort(average_last_rewards)[::-1]

names, _ = get_names_and_colors(prisoners)
print("The three best players are : ")
print('-'*30)
for i in best_ones[:3]:
  print(f"{i+1} :\t {names[i]},\t\t average final reward: {average_last_rewards[i]}" +
        f"\tstdev: {stdev_last_rewards[i]}")


### Uniform distribution in [-0.1, 1.1] (to include NICE and BAD as well)

In [ ]:
# @title
prisoners_number = 10

chrt = characteristics_list_generator(tit_tat_probability=0.2,
                                      pdf=scp.stats.uniform.pdf,
                                      loc=-0.1, scale=1.2)

prisoners = random_env_generator(chrt, n_prisoners)

data_rewards, data_actions = round_robin(prisoners, round_number=1)

#names and color for the animation
names, colors = get_names_and_colors(prisoners)

#animation
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

update_balls,  artists_balls  = rotating_players(
    fig, ax_left, data_actions, color=colors,
    moving_time=3, stop_time=9
)
update_reward, artists_reward = reward_time_animation(
    fig, ax_right, data_rewards, colors=colors, names=names,
    moving_time=3, stop_time=9
)

fig.tight_layout()

total_frames =12 * len(prisoners)

ani = combine_animations(
    fig         = fig,
    update_fns  = [update_balls, update_reward],
    artists     = [artists_balls, artists_reward],
    total_frames= total_frames,
    interval    = 150,
)

HTML(ani.to_jshtml())


In [ ]:
# @title
n_exp = 1000

average_last_rewards = np.zeros(prisoners_number)
stdev_last_rewards = np.zeros(prisoners_number)

for p in prisoners:
  p.total_reward=0

for round in range(n_exp):
  prisoners_exp = [ p.clone() for p in prisoners ]

  data_rewards, data_actions = round_robin(prisoners_exp, round_number=1)
  last_rewards = data_rewards[0,:,-1]
  average_last_rewards += last_rewards
  stdev_last_rewards += last_rewards**2

average_last_rewards /= n_exp
stdev_last_rewards = np.sqrt( (stdev_last_rewards/n_exp
                              - average_last_rewards**2) )
best_ones = np.argsort(average_last_rewards)[::-1]

names, _ = get_names_and_colors(prisoners)
print("The three best players are : ")
print('-'*30)
for i in best_ones[:3]:
  print(f"{i+1} :\t {names[i]},\t\t average final reward: {average_last_rewards[i]}" +
        f"\tstdev: {stdev_last_rewards[i]}")



The lesson we learn from this is that the most aggressive strategy (the one that defects the most) emerges as the dominant strategy in a single-round multiplayer Prisoner's Dilemma. This happens because defecting is the strategy with the highest payoff and in a single-round there's no interest in what the opponent does. But things change when we consider the multiple rounds case.

## Multiple Rounds case
We now study what happens when we reiterate the round robin multiple times

###One player for each personality, different k-values for MID

In [ ]:
# @title
#definining prisoners
prisoners = [ Prisoner("Nice",0),
              Prisoner("Mid",0.25),
              Prisoner("Mid",0.75),
              Prisoner("Mid",0.5),
              Prisoner("Tit-tat",0),
              Prisoner("Bad",1) ]

round_number = 20
data_rewards, data_actions = round_robin(prisoners, round_number=round_number)

#names and color for the animation
names, colors = get_names_and_colors(prisoners)

#animation
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

update_balls,  artists_balls  = rotating_players(
    fig, ax_left, data_actions, color=colors,
    moving_time=3, stop_time=9
)
update_reward, artists_reward = reward_time_animation(
    fig, ax_right, data_rewards, colors=colors, names=names,
    moving_time=3, stop_time=9
)

fig.tight_layout()

total_frames =12 * len(prisoners) * round_number

ani = combine_animations(
    fig         = fig,
    update_fns  = [update_balls, update_reward],
    artists     = [artists_balls, artists_reward],
    total_frames= total_frames,
    interval    = 150,
)

HTML(ani.to_jshtml())



In [ ]:
# @title
# Multiple round robin ranking
# if we repeat the experiment we obtain the same results on avarage
n_exp = 1000

prisoners_pop = [ Prisoner("Nice",0),
              Prisoner("Mid",0.25),
              Prisoner("Mid",0.75),
              Prisoner("Mid",0.5),
              Prisoner("Tit-tat",0),
              Prisoner("Bad",1) ]

round_number = 20

average_last_rewards = np.zeros(len(prisoners_pop))
stdev_last_rewards = np.zeros(len(prisoners_pop))

for round in range(n_exp):
  prisoners = [ p.clone() for p in prisoners_pop ]

  data_rewards, data_actions = round_robin(prisoners, round_number)
  last_rewards = data_rewards[-1,:,-1]
  average_last_rewards += last_rewards
  stdev_last_rewards += last_rewards**2

average_last_rewards /= n_exp
stdev_last_rewards = np.sqrt( (stdev_last_rewards/n_exp
                              - average_last_rewards**2))
best_ones = np.argsort(average_last_rewards)[::-1]

names, _ = get_names_and_colors(prisoners_pop)
print("The three best players are : ")
print('-'*30)
for i in best_ones:
  print(f"{i+1} :\t {names[i]},\t\t average final reward: {average_last_rewards[i]}" +
        f"\tstdev: {stdev_last_rewards[i]}")



As in the Two-Player and in the single round Multiple Players Prisoner's Dilemma, defecting is the dominant strategy.  
But what happens when we remove the MID personalities?
If we are in the two-players case we can see that the tit-tat can actually win. This happens because in a Round-Robin scheme each player also play with himself, so for the tit-tat cooperating with himself is the winning strategy. This happens in the multiple rounds case as well. We now study the results of a Round-Robin with a population composed by BAD and TIT-TAT only with different proportions.

###BAD and TIT-TAT, different proportions

In [ ]:
n_players = 10
n_rounds = 20
p_bad_list = np.linspace(0.1, 0.9, 9, endpoint=True)
crossing_match = []

for p_bad in p_bad_list:

    prisoners = nonRandom_env_generator(
        [[p_bad, "Bad", 1], [1-p_bad, "Tit-tat", 0]],
        n_players)

    data_rewards, _ = round_robin(prisoners, round_number=n_rounds)

    rewards_time = []
    for r in range(data_rewards.shape[0]):
        for j in range(data_rewards.shape[2]):
            rewards_time.append(data_rewards[r, :, j])
    rewards_time = np.array(rewards_time) # shape(n_match_tot, players)
    # it's the reward for each match for each player

    # to get indices of bad and tit-tat players
    names = [p.inv_dict[p.personality] for p in prisoners]
    tt_idx = [i for i, n in enumerate(names) if n == "Tit-tat"]
    bad_idx = [i for i, n in enumerate(names) if n == "Bad"]

    # average of the rewards of tit-tat (or bad) for every match
    mean_tt = np.mean(rewards_time[:, tt_idx], axis=1)
    mean_bad = np.mean(rewards_time[:, bad_idx], axis=1)
    # because I want to know when tit-tat ON AVARAGE become better than bad

    best_match = None

    for m in range(len(rewards_time)): # m match index
        if np.all(mean_tt[m:] > mean_bad[m:]):
          # the average reward for all tit-tat is greater than the one
          # for bad from the m match
            best_match = m
            break

    if best_match is None:
        best_match = len(rewards_time)

    crossing_match.append(best_match)


plt.figure(figsize=(8,5))
plt.plot(p_bad_list, crossing_match, marker='o', label="Crossing time")
if crossing_match[-1] == len(rewards_time):
    plt.scatter(p_bad_list[-1], crossing_match[-1], s=100, color='red', label="Never overtake")
plt.xlabel("Fraction of Bad players in the population")
plt.ylabel("Crossing match based on average reward (Tit-for-Tat > Bad)")
plt.title("Tit-tat overtake Bad (on average)")
plt.grid(True)
plt.legend()
plt.show()

We see that in the Multiple Rounds case with only TIT-TAT and BAD cooperating for tit-tat become the winning strategy.

#**3 - Evolution of Strategies in the Repeated MPIPD**
We now want to study how the population composition changes over a number of round robins, based on the reward obtained by each player.

Players with higher rewards survive into the next generation, and we observe which strategies prove to be the fittest.


For this purpose we developed the function `evolution`, which will also be used in the next section, but for now we disable strategy mutation (we simply control the `best_fraction` parameter and set `keep_best` to one, while `mutation` and `mutation_factor` are set to zero).

A detailed discussion of this function will be provided in the next section.

### First Test
Here you find our first test of the function. The number of round robins performed for each generation is set to 7. This number is not arbitrary... for this number of round robins you have a great variance on the population's final composition (if you run it multiple times you can see that the result can cange a lot).


In [ ]:
# @title
Prisoner.general_id = 0

#defining the type of prisoners we want to use
chrt = [[0.25,"Nice",0],
        [0.25,"Mid",0.5],
        [0.25,"Bad",1],
        [0.25,"Tit-tat",0.25]]
#creating 100 prisoners
prisoner_list = nonRandom_env_generator(chrt, 100)

def get_gen(prisoner_list):
    gen = {}
    for p in prisoner_list:
        name = get_names_and_colors([p])[0][0]
        gen[name] = gen.get(name, 0) + 1
    return gen

personality_vs_gen = []
personality_vs_gen.append(get_gen(prisoner_list))

# we evolve the population
generations_number = 13
for t in range(generations_number):
    Prisoner.general_id = 0
    best_fraction = 0.15 + 0.7 * np.exp(-t * 0.05)
    prisoner_list = evolution(prisoner_list, round_number=7,
                              best_fraction=best_fraction, keep_best=1,
                              mutation=0, heredity=0)
    personality_vs_gen.append(get_gen(prisoner_list))
    print(f"t={t}: best_fraction={best_fraction:.2f} "
          f"survivors={int(100*best_fraction)}")

df = pd.DataFrame(data=personality_vs_gen).fillna(0)

ax = sns.lineplot(data=df)
ax.set_xlabel("Generation")
ax.set_ylabel("Number of Players")
ax.grid(alpha=0.2)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=df.columns.tolist(), title='Personality Survival',
          loc='center left', bbox_to_anchor=(1, 0.5))

### Final Composition vs Number of Round Robins
(saved to file as results_complete)

We now want to see how the number of rounds for each iteration affects the survival of the various strategies. We divide the population equally in 4 strategies, Nice, Mid, Bad and Tit-for-Tat, for a total of 100. Then we evolve the system changing in each simulation the number of rounds between each generation, to see how the population fares.

In [ ]:
# @title
rounds_per_gen_list = [3, 5, 6, 7, 8, 9, 11]

n_simulations = 20  #number of simulations for each run with setted params
generations_number = 8 #number of generations to be studied

chrt = [[0.25,"Nice",0],
        [0.25,"Mid",0.5],
        [0.25,"Bad",1],
        [0.25,"Tit-tat",0.25]]
prisoners = nonRandom_env_generator(chrt, 100)


results = {r: [] for r in rounds_per_gen_list}

for r in rounds_per_gen_list:
    for _ in range(n_simulations):
        prisoner_list = [p.clone() for p in prisoners]
        df = simulate_evolution(
                prisoner_list_init= prisoner_list,
                rounds_per_gen= r,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)
        results[r].append(df)


In [ ]:
# @title

with open("results_complete.pkl", "rb") as f:
    results = pickle.load(f)

rounds_per_gen_list = [3, 5, 6, 7, 8, 9, 11]
names, colors = get_names_and_colors([
    Prisoner("Nice", 0), Prisoner("Bad", 0),
    Prisoner("Tit-tat", 0), Prisoner("Mid", 0.5)
])

fig, axes = plt.subplots(len(names), len(rounds_per_gen_list),
                         figsize=(4*len(rounds_per_gen_list), 3*len(names)),
                         sharey='row', sharex=True)

for col, r in enumerate(rounds_per_gen_list):
    dfs = results[r]  # list of df for this value of r
    for row, (strategy, color) in enumerate(zip(names, colors)):
        ax = axes[row, col] #plot the results in the right plot

        # estrai la colonna della strategia da ogni simulazione (0 se assente)
        data = np.array([df.get(strategy, pd.Series(0, index=df.index)).values
                         for df in dfs])  # shape: (n_simulations, generations+1)

        mean = data.mean(axis=0)
        std  = data.std(axis=0)
        gens = np.arange(generations_number + 1)

        ax.fill_between(gens, mean - std, mean + std, alpha=0.3, color=color)
        ax.plot(gens, mean, label="Average population", color=color)
        ax.plot(gens, mean - std, "k--", label="Population's stdev", alpha=0.3)
        ax.plot(gens, mean + std, "k--", alpha=0.3)
        ax.grid()
        ax.legend(loc='best')
        if row == 0:
            ax.set_title(f"{r} rounds/gen")
        if col == 0:
            ax.set_ylabel(strategy)

fig.supxlabel("Generation")
fig.supylabel("Number of individuals")
plt.tight_layout()
plt.show()

### Final Composition vs Defect Percentage and Initial Tit-tat Composition Fraction
(8 round robin per generation)

(saved to file results_k_vs_initial_frac_all-grid.pkl)

**All the grid**

In [ ]:
# @title
rounds_per_gen = 8

n_simulations = 20  #number of simulations for each run with setted params
generations_number = 8 #number of generations to be studied

k_grid = np.linspace(0.0, 1, 10)
composition_grid = np.linspace(0.0, 1, 10)

results = {(k, composition): [] for k in k_grid for composition in composition_grid}

#-----------------------------
# A unaccuratre estimation for execution
#-----------------------------------
start = time.time()
chrt = [[0.5,"Mid",0.5],
              [0.5,"Tit-tat",0.25]]
prisoners = nonRandom_env_generator(chrt, 20)
df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= rounds_per_gen,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)

single_time = time.time() - start

print(f"Expected time for execution: {single_time*n_simulations*len(k_grid)*len(composition_grid)/60} min")
#-------------------------------------------------------

for k in k_grid:
  for composition in composition_grid:
    for _ in range(n_simulations):
      chrt = [[1-composition,"Mid",k],
              [composition,"Tit-tat",0.25]]
      prisoners = nonRandom_env_generator(chrt, 20)
      df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= rounds_per_gen,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)
      results[(k, composition)].append(df)


In [ ]:
# @title
# heatmap
with open("results_k_vs_initial_frac_all-grid.pkl", "rb") as f:
    results = pickle.load(f)

k_grid = np.linspace(0.0, 1, 10)
composition_grid = np.linspace(0.0, 1, 10)
mean_matrix = np.zeros((len(k_grid), len(composition_grid)))
std_matrix  = np.zeros((len(k_grid), len(composition_grid)))

for i, k in enumerate(k_grid):
    for j, composition in enumerate(composition_grid):
        dfs = results[(k, composition)]
        tit_tat_final = []
        for df in dfs:
            final_gen = df.iloc[-1]
            total = final_gen.sum()
            # somma tutte le colonne Tit-tat (il nome è fisso)
            tit_tat = final_gen.get("Tit-tat", 0) / total
            tit_tat_final.append(tit_tat)
        mean_matrix[i, j] = np.mean(tit_tat_final)
        std_matrix[i, j]  = np.std(tit_tat_final)

# plot affiancato
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_labels = [f"{c:.1f}" for c in composition_grid]
y_labels  = [f"{k:.1f}" for k in k_grid]

sns.heatmap(mean_matrix,
            xticklabels=x_labels, yticklabels=y_labels,
            annot=True, fmt=".2f", cmap="coolwarm", vmin=0, vmax=1,
            ax=axes[0])
axes[0].set_xlabel("Initial Tit-tat fraction")
axes[0].set_ylabel("Mid defect probability (k)")
axes[0].set_title("Tit-tat fraction at final generation — mean")

sns.heatmap(std_matrix,
            xticklabels=x_labels, yticklabels=y_labels,
            annot=True, fmt=".2f", cmap="Oranges", vmin=0,
            ax=axes[1])
axes[1].set_xlabel("Initial Tit-tat fraction")
axes[1].set_ylabel("Mid defect probability (k)")
axes[1].set_title("Tit-tat fraction at final generation — std")

plt.tight_layout()
plt.show()

**Zoom**

(8 round robin per generation)

(saved to file results_k_vs_initial_frac_zoom-grid.pkl)

In [ ]:
# @title
rounds_per_gen = 8

n_simulations = 20  #number of simulations for each run with setted params
generations_number = 8 #number of generations to be studied

k_grid = np.linspace(0.01, 1, 10)
composition_grid = np.linspace(0.05, 0.4, 10)

results = {(k, composition): [] for k in k_grid for composition in composition_grid}

import time
start = time.time()
chrt = [[0.5,"Mid",0.5],
              [0.5,"Tit-tat",0.25]]
prisoners = nonRandom_env_generator(chrt, 20)
df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= rounds_per_gen,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)

single_time = time.time() - start

print(f"Expected time for execution: {single_time*n_simulations*len(k_grid)*len(composition_grid)/60}")
for k in k_grid:
  for composition in composition_grid:
    for _ in range(n_simulations):
      chrt = [[1-composition,"Mid",k],
              [composition,"Tit-tat",0.25]]
      prisoners = nonRandom_env_generator(chrt, 20)
      df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= rounds_per_gen,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)
      results[(k, composition)].append(df)


In [ ]:
# @title
with open("results_k_vs_initial_frac_zoom-grid.pkl", "rb") as f:
    results = pickle.load(f)

k_grid = np.linspace(0.01, 1, 10)
composition_grid = np.linspace(0.05, 0.4, 10)
mean_matrix = np.zeros((len(k_grid), len(composition_grid)))
std_matrix  = np.zeros((len(k_grid), len(composition_grid)))

for i, k in enumerate(k_grid):
    for j, composition in enumerate(composition_grid):
        dfs = results[(k, composition)]
        tit_tat_final = []
        for df in dfs:
            final_gen = df.iloc[-1]
            total = final_gen.sum()
            # somma tutte le colonne Tit-tat (il nome è fisso)
            tit_tat = final_gen.get("Tit-tat", 0) / total
            tit_tat_final.append(tit_tat)
        mean_matrix[i, j] = np.mean(tit_tat_final)
        std_matrix[i, j]  = np.std(tit_tat_final)

# plot affiancato
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_labels = [f"{c:.1f}" for c in composition_grid]
y_labels  = [f"{k:.1f}" for k in k_grid]

sns.heatmap(mean_matrix,
            xticklabels=x_labels, yticklabels=y_labels,
            annot=True, fmt=".2f", cmap="coolwarm", vmin=0, vmax=1,
            ax=axes[0])
axes[0].set_xlabel("Initial Tit-tat fraction")
axes[0].set_ylabel("Mid defect probability (k)")
axes[0].set_title("Tit-tat fraction at final generation — mean")

sns.heatmap(std_matrix,
            xticklabels=x_labels, yticklabels=y_labels,
            annot=True, fmt=".2f", cmap="Oranges", vmin=0,
            ax=axes[1])
axes[1].set_xlabel("Initial Tit-tat fraction")
axes[1].set_ylabel("Mid defect probability (k)")
axes[1].set_title("Tit-tat fraction at final generation — std")

plt.tight_layout()
plt.show()

In [ ]:
# @title
rows = []

k_grid = np.linspace(0.01, 1, 10)
composition_grid = np.linspace(0.05, 0.4, 10)
for k in k_grid:
    for composition in composition_grid:
        for df in results[(k, composition)]:
            final_gen = df.iloc[-1]
            total = final_gen.sum()
            tit_tat = final_gen.get("Tit-tat", 0) / total
            mid = sum(final_gen.get(c, 0) for c in final_gen.index
                      if c.startswith("Mid") or c.startswith("Bad") or c.startswith("Nice")) / total
            rows.append({
                "Mid's Defect Percentage": k,
                "Initial Tit-tat fraction": composition,
                "Final Tit-Tat fraction"     : tit_tat,
                "Final Mid Fraction"         : mid
            })

corr_df = pd.DataFrame(rows)

# matrice di correlazione
corr = corr_df.corr()

# heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()

### **Does anything change for a different number of round robins per generation? (let's say 15)**
(saved to file results_k_vs_initial_frac_15round.pkl)

In [ ]:
# @title
rounds_per_gen = 15

n_simulations = 20  #number of simulations for each run with setted params
generations_number = 8 #number of generations to be studied

k_grid = np.linspace(0.0, 1, 10)
composition_grid = np.linspace(0.0, 1, 10)

results = {(k, composition): [] for k in k_grid for composition in composition_grid}

#-----------------------------
# A unaccuratre estimation for execution
#-----------------------------------
start = time.time()
chrt = [[0.5,"Mid",0.5],
              [0.5,"Tit-tat",0.25]]
prisoners = nonRandom_env_generator(chrt, 20)
df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= rounds_per_gen,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)

single_time = time.time() - start

print(f"Expected time for execution: {single_time*n_simulations*len(k_grid)*len(composition_grid)/60}")
#-----------------------------------------

for k in k_grid:
  for composition in composition_grid:
    for _ in range(n_simulations):
      chrt = [[1-composition,"Mid",k],
              [composition,"Tit-tat",0.25]]
      prisoners = nonRandom_env_generator(chrt, 20)
      df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= rounds_per_gen,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)
      results[(k, composition)].append(df)


In [ ]:
# @title
with open("results_k_vs_initial_frac_15round.pkl", "rb") as f:
    results = pickle.load(f)

k_grid = np.linspace(0.0, 1, 10)
composition_grid = np.linspace(0.0, 1, 10)

mean_matrix = np.zeros((len(k_grid), len(composition_grid)))
std_matrix  = np.zeros((len(k_grid), len(composition_grid)))

for i, k in enumerate(k_grid):
    for j, composition in enumerate(composition_grid):
        dfs = results[(k, composition)]
        tit_tat_final = []
        for df in dfs:
            final_gen = df.iloc[-1]
            total = final_gen.sum()
            # somma tutte le colonne Tit-tat (il nome è fisso)
            tit_tat = final_gen.get("Tit-tat", 0) / total
            tit_tat_final.append(tit_tat)
        mean_matrix[i, j] = np.mean(tit_tat_final)
        std_matrix[i, j]  = np.std(tit_tat_final)

# plot affiancato
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_labels = [f"{c:.1f}" for c in composition_grid]
y_labels  = [f"{k:.1f}" for k in k_grid]

sns.heatmap(mean_matrix,
            xticklabels=x_labels, yticklabels=y_labels,
            annot=True, fmt=".2f", cmap="coolwarm", vmin=0, vmax=1,
            ax=axes[0])
axes[0].set_xlabel("Initial Tit-tat fraction")
axes[0].set_ylabel("Mid defect probability (k)")
axes[0].set_title("Tit-tat fraction at final generation — mean")

sns.heatmap(std_matrix,
            xticklabels=x_labels, yticklabels=y_labels,
            annot=True, fmt=".2f", cmap="Oranges", vmin=0,
            ax=axes[1])
axes[1].set_xlabel("Initial Tit-tat fraction")
axes[1].set_ylabel("Mid defect probability (k)")
axes[1].set_title("Tit-tat fraction at final generation — std")

plt.tight_layout()
plt.show()

### Studing the correlation with number of rounds as well
(saved to file results_k_vs_initial_frac_vs_round_num)

In [ ]:
# @title
rounds_per_gen_list = [3, 5, 6, 7, 8, 10, 15]

n_simulations = 20  #number of simulations for each run with setted params
generations_number = 8 #number of generations to be studied

k_grid = np.linspace(0.0, 1, 10)
composition_grid = np.linspace(0.0, 1, 10)

results = {(k, composition, round_robin): [] for k in k_grid
                                  for composition in composition_grid
                                  for round_robin in rounds_per_gen_list}

#-----------------------------
# A unaccuratre estimation for execution
#-----------------------------------
start = time.time()
chrt = [[0.5,"Mid",0.5],
              [0.5,"Tit-tat",0.25]]
prisoners = nonRandom_env_generator(chrt, 20)
df = simulate_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= 8,
                generations_number= generations_number,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)

single_time = time.time() - start

print(f"Expected time for execution: {single_time*n_simulations*len(k_grid)*len(composition_grid)*len(rounds_per_gen_list)/60} min")
#------------------------------------

for rounds_per_gen in rounds_per_gen_list:
  for k in k_grid:
    for composition in composition_grid:
      for _ in range(n_simulations):
        chrt = [[1-composition,"Mid",k],
                [composition,"Tit-tat",0.25]]
        prisoners = nonRandom_env_generator(chrt, 20)
        df = simulate_evolution(
                  prisoner_list_init= prisoners,
                  rounds_per_gen= rounds_per_gen,
                  generations_number= generations_number,
                  best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                  keep_best= 1, mutation= 0, heredity= 0,
                  k_par_mutation_factor= 0, personality_mutation_factor= 0)
        results[(k, composition, rounds_per_gen)].append(df)




In [ ]:
# @title
rows = []

with open("results_k_vs_initial_frac_vs_round_num.pkl", "rb") as f:
    results = pickle.load(f)

rounds_per_gen_list = [3, 5, 6, 7, 8, 10, 15]
k_grid = np.linspace(0.0, 1, 10)
composition_grid = np.linspace(0.0, 1, 10)

for round in rounds_per_gen_list:
  for k in k_grid[1:]:
    for composition in composition_grid[1:]:
        for df in results[(k, composition, round)]:
            final_gen = df.iloc[-1]
            total = final_gen.sum()
            tit_tat = final_gen.get("Tit-tat", 0) / total
            mid = sum(final_gen.get(c, 0) for c in final_gen.index
                      if c.startswith("Mid") or c.startswith("Bad") or c.startswith("Nice")) / total
            rows.append({
                "Mid's Defect Percentage": k,
                "Initial Tit-tat fraction": composition,
                "Round Robin's number": round,
                "Final Tit-Tat fraction": tit_tat,
                "Final Mid Fraction": mid
            })

corr_df = pd.DataFrame(rows)

corr = corr_df.corr()

# heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()

# **4 - Survival of the Fittest: Genetic Evolution in the MPIPD**

We have studied how the composition of the total population changes when strategies are fixed.

We now allow strategies to mutate by tuning the `mutation`, `mutation_factor` and `heredity` parameters of the `evolution` function, whose signature is:
```python
evolution(players_list, round_number, best_fraction,
          keep_best=0.4,
          mutation=0.3,
          k_par_mutation_factor=0.1, personality_mutation_factor=0.1,
          heredity=0.3)
```

We briefly describe how these parameters govern the creation of the next generation:

*   **Mutation** takes one of the survivors (a prisoner belonging to the best fraction set by `best_fraction`) and randomly modifies its parameters. The probability for the personality to change is controlled by `personality_mutation_factor` (Personality is sampled uniformly from {Mid, Tit-for-Tat}. Note that Nice and Bad are special cases of Mid (defect fraction 0 and 1 respectively) and are therefore not sampled separately.), while the intensity of the change in the defection probability (k parameter) is set by `k_par_mutation_factor`. This simulates random genetic mutations.

$$
k_{\text{new}} = k_{\text{parent}} \cdot [1 + \mathcal{N}(\mu=0,\ \sigma=k\_par\_mutation\_factor)]
$$

*   **Heredity** takes two survivors and assigns the new prisoner the personality of one of the two parents, chosen with equal probability. The defection probability of the offspring is the average of the parents' k parameters. This simulates genetic recombination during reproduction.

*   **Keep best** carries a fraction of the best survivors into the next generation with no changes to their parameters. This ensures that the fittest individuals are preserved across generations.

This model follows the same principles as the USPEX software, used for predicting stable crystal phases. The reference is the following [article](https://uspex-team.org/static/file/CPC-USPEX-2006.pdf).

## Drift in the *Bad* direction for *Mid* prisoners

Here we show that *Mid* prisoners tend to evolve toward a defection probability of 1 over time.
Mutation is enabled with `k_par_mutation_factor = 0.2`, while `personality_mutation_factor` is kept at 0.

In [ ]:
# @title
Prisoner.general_id = 0

#defining the type of prisoners we want to use
chrt = [[1,"Mid",0.5]]
#creating 100 prisoners
prisoner_list = nonRandom_env_generator(chrt, 100)

def get_gen(prisoner_list):
    gen = {}
    for p in prisoner_list:
        name = get_names_and_colors([p])[0][0]
        gen[name] = gen.get(name, 0) + 1
    return gen

personality_vs_gen = []
personality_vs_gen.append(get_gen(prisoner_list))

generations_number = 15
for t in range(generations_number):
    Prisoner.general_id = 0
    best_fraction = 0.15 + 0.7 * np.exp(-t * 0.05)
    prisoner_list = evolution(prisoner_list, round_number=7,
                              best_fraction=best_fraction, keep_best=0.2,
                              mutation=0.4, heredity=0.4,
                              k_par_mutation_factor=0.2,
                              personality_mutation_factor=0)
    personality_vs_gen.append(get_gen(prisoner_list))
    print(f"t={t}: best_fraction={best_fraction:.2f} "
          f"survivors={int(100*best_fraction)}")

df = pd.DataFrame(data=personality_vs_gen).fillna(0)

from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
fig, ax = plt.subplots(figsize=(8, 5))

#separing bad and mid column
columns = df.columns.tolist()
bad_cols = [c for c in columns if "Bad" in c]
mid_cols = [c for c in columns if "Mid" in c]

#to get k from column name"
def extract_k(col_name):

    for part in col_name.split():
        if part.endswith("%)"):
            return int(part[:-2]) / 100
    return 0


#colormap for Mid prisoners
cmap = cm.Blues
norm = mcolors.Normalize(vmin=0, vmax=1)

#plot mid
for col in bad_cols:
    ax.plot(df.index, df[col], color=cm.Reds(0.8), linewidth=2)

# plot for mid
for col in mid_cols:
    k = extract_k(col)
    ax.plot(df.index, df[col], color=cmap(norm(k)),
            linestyle="-", linewidth=1.5)

# color bar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Mid defect rate $k$")

# legend
legend_elements = [
    Line2D([0], [0], color=cm.Reds(0.8), linewidth=2, label="Bad"),
    Line2D([0], [0], color=cm.Blues(0.5), linewidth=1.5, linestyle="-", label="Mid"),
]
ax.legend(handles=legend_elements, loc="best")

ax.set_xlabel("Generation")
ax.set_ylabel("Number of Players")
ax.grid()
plt.tight_layout()
plt.show()

### How does `k_par_mutation_factor` affect the population's mean final value of k?

In [ ]:
# @title
rounds_per_gen = 10

n_simulations = 50  #number of simulations for each run with setted params
generations_number = [7, 10, 13, 16, 19] #number of generations to be studied


k_par_mutation_factor_grid = np.linspace(0, 1, 10)
(keep_best, mutation, heredity) = (0, 1, 0)

mean = []
std = []

x = []

chrt = [[1, "Mid", 0.01]]
prisoner_list = nonRandom_env_generator(chrt, 20)
import time
start = time.time()
prisoners = [p.clone() for p in prisoner_list]
mean_curr, std_curr = k_par_mean_std_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= 10,
                generations_number= generations_number[2],
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= 1, mutation= 0, heredity= 0,
                k_par_mutation_factor= 0, personality_mutation_factor= 0)

single_time = time.time() - start

print(f"Expected time for execution: {single_time*n_simulations*len(k_par_mutation_factor_grid)*len(generations_number)/60}")

for n_generations in generations_number:
  for k_par_mutation_factor in k_par_mutation_factor_grid:
      mean_curr=0
      std_curr=0
      for _ in range(n_simulations):
        prisoners = [p.clone() for p in prisoner_list]
        m, v = k_par_mean_std_evolution(
                prisoner_list_init= prisoners,
                rounds_per_gen= 10,
                generations_number= n_generations,
                best_fraction_func= lambda t: 0.15 + 0.7 * np.exp(-t * 0.05),
                keep_best= keep_best, mutation= mutation, heredity=heredity,
                k_par_mutation_factor= k_par_mutation_factor, personality_mutation_factor= 0)
        mean_curr += m[-1]
        std_curr += v[-1]
      mean_curr /= n_simulations
      std_curr /= n_simulations

      x.append([n_generations, k_par_mutation_factor])
      mean.append(mean_curr)
      std.append(std_curr)


In [ ]:
# @title
#sigmoidal fit function
with open("results_sigmoidal_fit_x.pkl", "rb") as f:
    x = pickle.load(f)
with open("results_sigmoidal_fit_mean.pkl", "rb") as f:
    mean = pickle.load(f)
with open("results_sigmoidal_fit_std.pkl", "rb") as f:
    std = pickle.load(f)

generations_number = [7, 10, 13, 16, 19]

def func_fit(x, b, c):
    return 1 / (np.exp(-(x + c) / b) + 1)


mean = np.array(mean)
std = np.array(std)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

#color map for different generation number
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(generations_number)))

#definining this to have smooth curves

k_smooth = np.linspace(0, 1, 200)
generations_number = [7, 10, 13, 16, 19]

for i, n_gen in enumerate(generations_number):
    mask = x[:, 0] == n_gen
    k_gen = x[mask, 1]
    mean_gen = mean[mask]
    std_gen = std[mask]

    #plotting data with errorbars
    ax.errorbar(k_gen, mean_gen, yerr=std_gen,
                fmt='o', color=colors[i], capsize=3,
                label=f'{n_gen} number of generation (data)', alpha=0.7)

    # fitting data of n_gen generation with fitting curve
    par, par_corr = scp.optimize.curve_fit(
        func_fit, k_gen, mean_gen, sigma=std_gen,
        p0=[0.09, 0.5], maxfev=10000
    )

    # plotting the fitting curve
    y_fit = func_fit(k_smooth, *par)
    ax.plot(k_smooth, y_fit, color=colors[i],
            label=f'{n_gen} number of generation (a={par[0]:.3f}, b={par[1]:.3f})')

#setting axes, legend, title ecc
ax.set_xlabel("k_par_mutation_factor")
ax.set_ylabel("Average last generation k_par")
ax.set_title("Sigmoidal fit")
ax.legend(bbox_to_anchor=(0.01, 1), loc='upper left', fontsize=8)
ax.grid()
plt.tight_layout()
plt.show()



### Nothing changes if we enable `personality_mutation_factor`. It just add more noise

In [ ]:
# @title
Prisoner.general_id = 0

#defining the type of prisoners we want to use
chrt = [[1,"Bad",0.5]]
#creating 100 prisoners
prisoner_list = nonRandom_env_generator(chrt, 100)

def get_gen(prisoner_list):
    gen = {}
    for p in prisoner_list:
        name = get_names_and_colors([p])[0][0]
        gen[name] = gen.get(name, 0) + 1
    return gen

personality_vs_gen = []
personality_vs_gen.append(get_gen(prisoner_list))

generations_number = 30
for t in range(generations_number):
    Prisoner.general_id = 0
    best_fraction = 0.15 + 0.7 * np.exp(-t * 0.05)
    prisoner_list = evolution(prisoner_list, round_number=7,
                              best_fraction=best_fraction, keep_best=0.2,
                              mutation=0.4, heredity=0.4,
                              k_par_mutation_factor=0.2,
                              personality_mutation_factor=0.5)
    personality_vs_gen.append(get_gen(prisoner_list))
    print(f"t={t}: best_fraction={best_fraction:.2f} "
          f"survivors={int(100*best_fraction)}")

df = pd.DataFrame(data=personality_vs_gen).fillna(0)

from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
fig, ax = plt.subplots(figsize=(8, 5))

#separing bad and mid column
columns = df.columns.tolist()
bad_cols = [c for c in columns if "Bad" in c]
mid_cols = [c for c in columns if "Mid" in c]

#to get k from column name"
def extract_k(col_name):

    for part in col_name.split():
        if part.endswith("%)"):
            return int(part[:-2]) / 100
    return 0


#colormap for Mid prisoners
cmap = cm.Blues
norm = mcolors.Normalize(vmin=0, vmax=1)

#plot mid
for col in bad_cols:
    ax.plot(df.index, df[col], color=cm.Reds(0.8), linewidth=2)

# plot for mid
for col in mid_cols:
    k = extract_k(col)
    ax.plot(df.index, df[col], color=cmap(norm(k)),
            linestyle="-", linewidth=1.5)

# color bar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Mid defect rate $k$")

# legend
legend_elements = [
    Line2D([0], [0], color=cm.Reds(0.8), linewidth=2, label="Bad"),
    Line2D([0], [0], color=cm.Blues(0.5), linewidth=1.5, linestyle="-", label="Mid"),
]
ax.legend(handles=legend_elements, loc="best")

ax.set_xlabel("Generation")
ax.set_ylabel("Number of Players")
ax.grid()
plt.tight_layout()
plt.show()